# Ablation Study về Regularization chống Overfitting (MLP từ Scratch)

Notebook này nghiên cứu hiện tượng **Overfitting** và cách các kỹ thuật **Regularization** giúp khắc phục nó, bằng cách hiện thực một mạng **MultiLayer Perceptron (MLP)** từ đầu bằng **NumPy**.

## 1. Bản chất của Overfitting

- Dữ liệu luôn có cả **đặc trưng tốt** (quy luật thật sự) và **đặc trưng xấu/nhiễu** (noise).
- Câu hỏi cốt lõi: **model học được đặc trưng nào của dữ liệu?**
- Khi model quá lớn / huấn luyện quá lâu, nó học luôn cả những đặc trưng xấu (nhiễu) → khớp hoàn hảo trên tập Train nhưng kém trên Validation/Test.

## 2. Các kỹ thuật Regularization được khảo sát

| Kỹ thuật | Ý tưởng cốt lõi |
| :--- | :--- |
| **L1 / L2** (thường dùng L2) | Phạt biên độ lớn của trọng số, giữ model đơn giản |
| **Dropout** | Tắt ngẫu nhiên một số nơ-ron, hạn chế co-adaptation |
| **Data Augmentation** | Thêm nhiễu vào dữ liệu để model học bất biến hơn |
| **Early Stopping** | Dừng sớm khi Validation Loss ngừng cải thiện |
| **Learning Rate Decay** | Giảm dần learning rate để hội tụ ổn định |

## 3. Phương pháp

**Tắt từng kỹ thuật một (ablation)** rồi tổng hợp thành bảng so sánh để thấy đóng góp của mỗi thành phần.

In [ ]:
# Bước 1: Import các thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Đã import thành công các thư viện!")

## 4. Định nghĩa mô hình MLP có tích hợp Regularization

Lớp `MLPClassifierWithRegularization` hỗ trợ đồng thời nhiều kỹ thuật regularization, mỗi kỹ thuật bật/tắt qua tham số khởi tạo:

- **L2**: cộng `l2_lambda * W` vào gradient (đạo hàm của `0.5 * λ * ||W||²`).
- **L1**: cộng `l1_lambda * sign(W)` vào gradient.
- **Dropout**: dùng *Inverted Dropout* trên các lớp ẩn khi training.
- **LR Decay**: `lr = init_lr / (1 + decay * epoch)`.
- **Early Stopping**: lưu lại trọng số tốt nhất theo Validation Loss, dừng khi quá `patience` epoch không cải thiện.

In [ ]:
# Bước 2: Định nghĩa lớp MLP với các kỹ thuật Regularization từ Scratch

class MLPClassifierWithRegularization:
    """MLP Classifier với các kỹ thuật regularization nâng cao, hiện thực từ đầu bằng NumPy."""

    def __init__(
        self,
        layer_sizes: list[int],
        learning_rate: float = 0.1,
        epochs: int = 2000,
        l1_lambda: float = 0.0,
        l2_lambda: float = 0.0,
        dropout_rate: float = 0.0,
        lr_decay: float = 0.0,
        patience: int = None,
    ):
        self.layer_sizes = layer_sizes
        self.init_lr = learning_rate
        self.lr = learning_rate
        self.epochs = epochs
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        self.lr_decay = lr_decay
        self.patience = patience

        # Khởi tạo trọng số (He Initialization)
        self.W: list[np.ndarray] = []
        self.b: list[np.ndarray] = []
        for i in range(1, len(layer_sizes)):
            limit = np.sqrt(2.0 / layer_sizes[i - 1])
            self.W.append(np.random.randn(layer_sizes[i], layer_sizes[i - 1]) * limit)
            self.b.append(np.zeros((layer_sizes[i], 1)))

        # Lưu mask của Dropout để dùng lại trong backward pass
        self.dropout_masks: list[np.ndarray] = []
        self.training = True

        # Lịch sử huấn luyện
        self.loss_history: list[float] = []
        self.val_loss_history: list[float] = []
        self.accuracy_history: list[float] = []
        self.val_accuracy_history: list[float] = []

        # Trọng số tốt nhất cho Early Stopping
        self.best_W: list[np.ndarray] = []
        self.best_b: list[np.ndarray] = []
        self.best_epoch = 0

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def _sigmoid_derivative(a: np.ndarray) -> np.ndarray:
        return a * (1.0 - a)

    def _forward(self, X: np.ndarray) -> tuple[list[np.ndarray], list[np.ndarray]]:
        """Lan truyền tiến với Inverted Dropout."""
        a_values = [X]
        z_values = []
        self.dropout_masks = []

        a = X
        num_layers = len(self.W)
        for l in range(num_layers):
            z = self.W[l] @ a + self.b[l]
            a = self._sigmoid(z)

            # Áp dụng Inverted Dropout cho lớp ẩn khi training
            if self.training and self.dropout_rate > 0.0 and l < num_layers - 1:
                mask = (np.random.rand(*a.shape) >= self.dropout_rate) / (1.0 - self.dropout_rate)
                a = a * mask
                self.dropout_masks.append(mask)
            else:
                self.dropout_masks.append(np.ones_like(a))

            z_values.append(z)
            a_values.append(a)

        return a_values, z_values

    def _backward(self, a_values: list[np.ndarray], y: np.ndarray) -> tuple[list[np.ndarray], list[np.ndarray]]:
        """Lan truyền ngược, gồm cả gradient của L1, L2 và mask Dropout."""
        n_samples = y.shape[1]
        dW = []
        db = []

        # Lỗi ở lớp đầu ra
        a_out = a_values[-1]
        delta = (a_out - y) * self._sigmoid_derivative(a_out)

        num_layers = len(self.W)
        for l in reversed(range(num_layers)):
            a_prev = a_values[l]

            # Gradient cơ bản
            dW_l = (1.0 / n_samples) * (delta @ a_prev.T)
            db_l = (1.0 / n_samples) * np.sum(delta, axis=1, keepdims=True)

            # Gradient của L2 Regularization: lambda * W
            if self.l2_lambda > 0.0:
                dW_l += self.l2_lambda * self.W[l]

            # Gradient của L1 Regularization: lambda * sign(W)
            if self.l1_lambda > 0.0:
                dW_l += self.l1_lambda * np.sign(self.W[l])

            dW.insert(0, dW_l)
            db.insert(0, db_l)

            if l > 0:
                # Lan truyền lỗi ngược và áp dụng mask dropout của lớp trước
                delta = (self.W[l].T @ delta) * self._sigmoid_derivative(a_prev)
                if self.training and self.dropout_rate > 0.0:
                    delta = delta * self.dropout_masks[l - 1]

        return dW, db

    def fit(self, X: np.ndarray, y: np.ndarray, X_val: np.ndarray = None, y_val: np.ndarray = None, augment: bool = False) -> "MLPClassifierWithRegularization":
        """Huấn luyện với tùy chọn Early Stopping, LR decay và Data Augmentation."""
        # Chuyển vị về dạng (n_features, n_samples) và (n_output, n_samples)
        X_input = X.T
        y_input = y.reshape(1, -1) if len(y.shape) == 1 else y.T

        if X_val is not None and y_val is not None:
            X_val_input = X_val.T
            y_val_input = y_val.reshape(1, -1) if len(y_val.shape) == 1 else y_val.T
        else:
            X_val_input = None
            y_val_input = None

        self.loss_history = []
        self.val_loss_history = []
        self.accuracy_history = []
        self.val_accuracy_history = []

        best_val_loss = float("inf")
        no_improvement_count = 0
        self.best_W = [np.copy(w) for w in self.W]
        self.best_b = [np.copy(bias) for bias in self.b]
        self.best_epoch = 0

        for epoch in range(1, self.epochs + 1):
            # Learning Rate Decay
            if self.lr_decay > 0.0:
                self.lr = self.init_lr / (1.0 + self.lr_decay * epoch)

            # Data Augmentation (thêm nhiễu Gaussian nhỏ vào input mỗi epoch)
            if augment:
                noise = np.random.normal(0, 0.05, X_input.shape)
                current_X = X_input + noise
            else:
                current_X = X_input

            # Forward pass (chế độ Training)
            self.training = True
            a_values, _ = self._forward(current_X)
            y_hat = a_values[-1]

            # Train Loss (kèm penalty L1 & L2)
            loss = 0.5 * np.mean(np.sum((y_hat - y_input) ** 2, axis=0))
            if self.l2_lambda > 0.0:
                loss += 0.5 * self.l2_lambda * sum(np.sum(w ** 2) for w in self.W)
            if self.l1_lambda > 0.0:
                loss += self.l1_lambda * sum(np.sum(np.abs(w)) for w in self.W)

            self.loss_history.append(loss)

            # Train Accuracy
            preds = (y_hat >= 0.5).astype(int)
            acc = float(np.mean(preds == y_input))
            self.accuracy_history.append(acc)

            # Backward pass & cập nhật tham số
            dW, db = self._backward(a_values, y_input)
            for l in range(len(self.W)):
                self.W[l] -= self.lr * dW[l]
                self.b[l] -= self.lr * db[l]

            # Đánh giá trên Validation (chế độ Evaluation - không dropout)
            if X_val_input is not None and y_val_input is not None:
                self.training = False
                a_val_values, _ = self._forward(X_val_input)
                y_val_hat = a_val_values[-1]

                val_loss = 0.5 * np.mean(np.sum((y_val_hat - y_val_input) ** 2, axis=0))
                if self.l2_lambda > 0.0:
                    val_loss += 0.5 * self.l2_lambda * sum(np.sum(w ** 2) for w in self.W)
                if self.l1_lambda > 0.0:
                    val_loss += self.l1_lambda * sum(np.sum(np.abs(w)) for w in self.W)

                self.val_loss_history.append(val_loss)

                val_preds = (y_val_hat >= 0.5).astype(int)
                val_acc = float(np.mean(val_preds == y_val_input))
                self.val_accuracy_history.append(val_acc)

                # Logic Early Stopping
                if self.patience is not None:
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        self.best_W = [np.copy(w) for w in self.W]
                        self.best_b = [np.copy(bias) for bias in self.b]
                        self.best_epoch = epoch
                        no_improvement_count = 0
                    else:
                        no_improvement_count += 1

                    if no_improvement_count >= self.patience:
                        # Khôi phục trọng số tốt nhất và dừng
                        self.W = [np.copy(w) for w in self.best_W]
                        self.b = [np.copy(bias) for bias in self.best_b]
                        break

        # Nếu không dùng/không kích hoạt Early Stopping, cập nhật trọng số tốt nhất là trọng số cuối
        if self.patience is None or no_improvement_count < self.patience:
            self.best_W = [np.copy(w) for w in self.W]
            self.best_b = [np.copy(bias) for bias in self.b]
            self.best_epoch = epoch

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Dự đoán xác suất (chế độ Evaluation)."""
        self.training = False
        X_input = X.T
        a_values, _ = self._forward(X_input)
        return a_values[-1].T

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Dự đoán nhãn lớp (chế độ Evaluation)."""
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int).ravel()

print("Đã định nghĩa thành công lớp MLPClassifierWithRegularization!")

## 5. Chuẩn bị dữ liệu dễ Overfit

Để overfitting **lộ rõ cả ở Test Accuracy** (không chỉ ở Gap), ta thiết kế dữ liệu như sau:

- **Nhiễu vừa phải** (`noise=0.25`): vẫn có **quy luật thật** để học (model tốt đạt ~90%+), nhưng đủ nhiễu để model tham lam học vẹt.
- **Train RẤT nhỏ** (chỉ **120 mẫu**) so với mạng lớn `[2, 64, 32, 1]` (~2300 trọng số) → ép overfit mạnh.
- **Test RẤT lớn** (**1260 mẫu**) → mỗi điểm chỉ chiếm ~0.08%, nên Test Accuracy đủ **mịn** để phân biệt khác biệt nhỏ (khắc phục lỗi "Test Acc đứng yên" của setup nhỏ trước).

Chia: **120 Train / 120 Val / 1260 Test** và chuẩn hóa đặc trưng.

In [ ]:
# Bước 3: Tạo và phân tách dữ liệu (Train NHỎ + Test LỚN để thấy rõ overfitting)
np.random.seed(42)

# Dataset lớn, nhiễu vừa phải -> có quy luật thật để học, nhưng vẫn dễ overfit khi train ít
X, y = make_moons(n_samples=1500, noise=0.25, random_state=42)

# Lấy 240 mẫu cho Train+Val, phần còn lại (1260 mẫu) làm Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, train_size=240, random_state=42)
# Chia đôi 240 -> 120 Train / 120 Val
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

## 6. Thiết lập các cấu hình Ablation

Ý tưởng: bắt đầu từ cấu hình **"All Regularizations"** (bật tất cả), rồi **tắt từng kỹ thuật một** để đo tác động riêng của nó. Thêm một **Baseline** hoàn toàn không regularization để làm mốc Overfitting.

In [ ]:
# Bước 4: Định nghĩa cấu hình huấn luyện
layer_sizes = [2, 64, 32, 1]
max_epochs = 3000

# "All Regularizations" bật TẤT CẢ (gồm cả L1). Mỗi "No X" tắt đúng một kỹ thuật.
configs = {
    "Baseline (Overfitted)": {
        "l1_lambda": 0.0, "l2_lambda": 0.0, "dropout_rate": 0.0,
        "lr_decay": 0.0, "patience": None, "augment": False, "lr": 0.15
    },
    "All Regularizations": {
        "l1_lambda": 0.0005, "l2_lambda": 0.002, "dropout_rate": 0.25,
        "lr_decay": 0.0005, "patience": 150, "augment": True, "lr": 0.15
    },
    "No L1 Regularization": {
        "l1_lambda": 0.0, "l2_lambda": 0.002, "dropout_rate": 0.25,
        "lr_decay": 0.0005, "patience": 150, "augment": True, "lr": 0.15
    },
    "No L2 Regularization": {
        "l1_lambda": 0.0005, "l2_lambda": 0.0, "dropout_rate": 0.25,
        "lr_decay": 0.0005, "patience": 150, "augment": True, "lr": 0.15
    },
    "No Dropout": {
        "l1_lambda": 0.0005, "l2_lambda": 0.002, "dropout_rate": 0.0,
        "lr_decay": 0.0005, "patience": 150, "augment": True, "lr": 0.15
    },
    "No Data Augmentation": {
        "l1_lambda": 0.0005, "l2_lambda": 0.002, "dropout_rate": 0.25,
        "lr_decay": 0.0005, "patience": 150, "augment": False, "lr": 0.15
    },
    "No Early Stopping": {
        "l1_lambda": 0.0005, "l2_lambda": 0.002, "dropout_rate": 0.25,
        "lr_decay": 0.0005, "patience": None, "augment": True, "lr": 0.15
    },
    "No LR Decay": {
        "l1_lambda": 0.0005, "l2_lambda": 0.002, "dropout_rate": 0.25,
        "lr_decay": 0.0, "patience": 150, "augment": True, "lr": 0.15
    }
}

print(f"Đã thiết lập {len(configs)} cấu hình ablation.")

## 7. Huấn luyện toàn bộ cấu hình

In [ ]:
# Bước 5: Chạy Ablation Study
results = {}

print("=" * 70)
print("  Bắt đầu chạy Ablation Study về Regularization trên Moons Dataset")
print("=" * 70)

for name, cfg in configs.items():
    print(f"\nHuấn luyện cấu hình: {name} ...")

    mlp = MLPClassifierWithRegularization(
        layer_sizes=layer_sizes,
        learning_rate=cfg["lr"],
        epochs=max_epochs,
        l1_lambda=cfg["l1_lambda"],
        l2_lambda=cfg["l2_lambda"],
        dropout_rate=cfg["dropout_rate"],
        lr_decay=cfg["lr_decay"],
        patience=cfg["patience"]
    )

    mlp.fit(X_train, y_train, X_val, y_val, augment=cfg["augment"])

    # Đánh giá trên train/val/test tại epoch tốt nhất
    train_acc = mlp.accuracy_history[mlp.best_epoch - 1]
    val_acc = mlp.val_accuracy_history[mlp.best_epoch - 1] if X_val is not None else 0.0

    test_preds = mlp.predict(X_test)
    test_acc = float(np.mean(test_preds == y_test))

    overfitting_gap = train_acc - val_acc

    print(f"--> Hoàn thành sau {mlp.best_epoch} epochs | Test Acc: {test_acc:.2%} | Val Acc: {val_acc:.2%} | Gap: {overfitting_gap:.2%}")

    results[name] = {
        "epochs_run": mlp.best_epoch,
        "train_loss": mlp.loss_history[mlp.best_epoch - 1],
        "val_loss": mlp.val_loss_history[mlp.best_epoch - 1] if X_val is not None else 0.0,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "overfitting_gap": overfitting_gap,
        "model": mlp
    }

## 8. Bảng so sánh kết quả

Tổng hợp kết quả thành bảng. Cột quan trọng nhất là **Overfitting Gap (Train - Val)**: gap càng nhỏ thì model càng tổng quát hóa tốt.

In [ ]:
# Bước 6: In bảng so sánh
header = f"| {'Cấu hình':<24} | {'Epochs':>6} | {'Train Loss':>10} | {'Val Loss':>9} | {'Train Acc':>9} | {'Val Acc':>8} | {'Test Acc':>8} | {'Gap':>7} |"
sep = "|" + "-" * (len(header) - 2) + "|"
print(header)
print(sep)
for name, d in results.items():
    print(
        f"| {name:<24} | {d['epochs_run']:>6} | {d['train_loss']:>10.5f} | {d['val_loss']:>9.5f} | "
        f"{d['train_acc']:>8.2%} | {d['val_acc']:>7.2%} | {d['test_acc']:>7.2%} | {d['overfitting_gap']:>6.2%} |"
    )

## 9. Trực quan hóa: Loss & Accuracy theo Epoch

So sánh đường cong huấn luyện của **Baseline** (overfit) và **All Regularizations** để thấy rõ khoảng cách Train–Val.

In [ ]:
# Bước 7: Vẽ đường cong Loss / Accuracy
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for name in ["Baseline (Overfitted)", "All Regularizations"]:
    mlp = results[name]["model"]
    axes[0].plot(mlp.loss_history, label=f"{name} - Train", linewidth=2.0)
    axes[0].plot(mlp.val_loss_history, label=f"{name} - Val", linestyle="--", linewidth=2.0)
    axes[1].plot(mlp.accuracy_history, label=f"{name} - Train", linewidth=2.0)
    axes[1].plot(mlp.val_accuracy_history, label=f"{name} - Val", linestyle="--", linewidth=2.0)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("So sánh Loss: Baseline vs All Regularizations")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("So sánh Accuracy: Baseline vs All Regularizations")
axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Trực quan hóa: Ranh giới quyết định (Decision Boundary)

Đây là minh chứng trực quan nhất cho Overfitting: ranh giới của Baseline **uốn lượn để khớp cả nhiễu**, còn ranh giới của model có regularization **trơn tru** hơn.

In [ ]:
# Bước 8: Vẽ ranh giới quyết định
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

mlp_baseline = results["Baseline (Overfitted)"]["model"]
mlp_regularized = results["All Regularizations"]["model"]

# Lưới điểm
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = scaler.transform(grid)

# Tọa độ điểm Test trong không gian gốc (đảo ngược StandardScaler)
X_test_orig = X_test * scaler.scale_ + scaler.mean_

# Baseline
probs_base = mlp_baseline.predict_proba(grid_scaled).reshape(xx.shape)
axes[0].contourf(xx, yy, probs_base, levels=50, cmap="RdYlBu", alpha=0.6)
axes[0].scatter(X_test_orig[:, 0], X_test_orig[:, 1], c=y_test, cmap="RdYlBu", edgecolors="black", linewidth=0.7, s=45)
axes[0].set_title(f"Baseline (Overfitted) - Test Acc: {results['Baseline (Overfitted)']['test_acc']:.2%}\n(Ranh giới uốn lượn, khớp cả nhiễu)")
axes[0].grid(True, alpha=0.3)

# Regularized
probs_reg = mlp_regularized.predict_proba(grid_scaled).reshape(xx.shape)
axes[1].contourf(xx, yy, probs_reg, levels=50, cmap="RdYlBu", alpha=0.6)
axes[1].scatter(X_test_orig[:, 0], X_test_orig[:, 1], c=y_test, cmap="RdYlBu", edgecolors="black", linewidth=0.7, s=45)
axes[1].set_title(f"All Regularizations - Test Acc: {results['All Regularizations']['test_acc']:.2%}\n(Ranh giới trơn tru, chống nhiễu)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Nhận xét & Kết luận

1. **Baseline (Overfitted)**: Không có regularization, mô hình chạy đủ 3000 epoch để khớp toàn bộ dữ liệu huấn luyện (Train Acc ~100%), nhưng kém nhất trên Validation/Test do khớp cả các điểm nhiễu → **Overfitting Gap lớn nhất**.
2. **All Regularizations**: Giảm Overfitting Gap đáng kể nhất, giữ ranh giới quyết định trơn tru và đạt độ chính xác kiểm thử cao hơn rõ rệt.
3. **Dropout & L2** là hai nhân tố quan trọng nhất: L2 kiểm soát biên độ trọng số, Dropout hạn chế co-adaptation của nơ-ron. Tắt một trong hai → Gap tăng mạnh ngay.
4. **Early Stopping** tiết kiệm lớn tài nguyên tính toán (dừng ở vài trăm epoch thay vì 3000) và ngăn Validation Loss tăng ngược ở nửa sau quá trình huấn luyện.
5. **Data Augmentation & LR Decay** đóng vai trò bổ trợ, giúp quá trình huấn luyện ổn định và tổng quát hóa tốt hơn.

> **Key takeaway:** Mục tiêu của regularization là buộc model học **đặc trưng tốt** (quy luật thật) thay vì ghi nhớ **đặc trưng xấu** (nhiễu) trong dữ liệu.

## 12. So sánh trực tiếp: DÙNG vs KHÔNG DÙNG từng kỹ thuật

Phần trên là *ablation* (bật hết rồi tắt từng cái). Phần này làm ngược lại để thấy rõ **"có cái đó thì khác gì"**:

- **Không dùng** = baseline trắng, tắt toàn bộ regularization.
- **Dùng** = baseline trắng **chỉ bật riêng một kỹ thuật**.

Để công bằng, ta **reset seed trước mỗi lần train** → trọng số khởi tạo y hệt nhau, khác biệt duy nhất là kỹ thuật được bật.

Hai cột cần để ý:
- **Δ Test Acc** = Test(Dùng) − Test(Không) → muốn **dương** (tổng quát hóa tốt hơn).
- **Δ Gap** = Gap(Dùng) − Gap(Không) → muốn **âm** (giảm overfitting).

In [ ]:
# Bước 9: Train cô lập từng kỹ thuật (reset seed để chỉ khác đúng kỹ thuật đang xét)
def train_eval(seed: int = 42, augment: bool = False, **kwargs) -> dict:
    np.random.seed(seed)  # khởi tạo trọng số giống hệt nhau giữa các lần
    mlp = MLPClassifierWithRegularization(
        layer_sizes=layer_sizes, learning_rate=0.15, epochs=max_epochs, **kwargs
    )
    mlp.fit(X_train, y_train, X_val, y_val, augment=augment)
    e = mlp.best_epoch
    train_acc = mlp.accuracy_history[e - 1]
    val_acc = mlp.val_accuracy_history[e - 1]
    test_acc = float(np.mean(mlp.predict(X_test) == y_test))
    return {"epochs": e, "train_acc": train_acc, "val_acc": val_acc,
            "test_acc": test_acc, "gap": train_acc - val_acc}

# Mỗi kỹ thuật: chỉ bật đúng tham số tương ứng (đã thêm L1)
techniques = {
    "L1 Regularization": {"l1_lambda": 0.0005},
    "L2 Regularization": {"l2_lambda": 0.002},
    "Dropout":           {"dropout_rate": 0.25},
    "Data Augmentation": {"augment": True},
    "Early Stopping":    {"patience": 150},
    "LR Decay":          {"lr_decay": 0.0005},
}

base = train_eval()  # KHÔNG dùng gì — mốc chung cho mọi so sánh
pair_results = {name: {"off": base, "on": train_eval(**kw)} for name, kw in techniques.items()}
print(f"Baseline (Không dùng gì): Test Acc = {base['test_acc']:.2%} | Gap = {base['gap']:.2%}")
print("Đã train xong tất cả cấu hình cô lập.")

In [ ]:
# Bước 10: Bảng DÙNG vs KHÔNG DÙNG
header = (f"| {'Kỹ thuật':<20} | {'Test Acc (Không)':>16} | {'Test Acc (Dùng)':>15} | {'Δ Test':>8} | "
         f"{'Gap (Không)':>11} | {'Gap (Dùng)':>10} | {'Δ Gap':>8} |")
print(header)
print("|" + "-" * (len(header) - 2) + "|")
for name, r in pair_results.items():
    off, on = r["off"], r["on"]
    d_test = on["test_acc"] - off["test_acc"]
    d_gap = on["gap"] - off["gap"]
    print(
        f"| {name:<20} | {off['test_acc']:>15.2%} | {on['test_acc']:>14.2%} | {d_test:>+7.2%} | "
        f"{off['gap']:>10.2%} | {on['gap']:>9.2%} | {d_gap:>+7.2%} |"
    )

print("\nGhi chú: Δ Test dương = tốt hơn | Δ Gap âm = giảm overfitting.")

In [ ]:
# Bước 11: Biểu đồ cột so sánh Dùng vs Không dùng
names = list(pair_results.keys())
off_test = [pair_results[n]["off"]["test_acc"] for n in names]
on_test = [pair_results[n]["on"]["test_acc"] for n in names]
off_gap = [pair_results[n]["off"]["gap"] for n in names]
on_gap = [pair_results[n]["on"]["gap"] for n in names]

x = np.arange(len(names))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(x - w / 2, off_test, w, label="Không dùng", color="#d95f5f")
axes[0].bar(x + w / 2, on_test, w, label="Dùng", color="#5f9ed9")
axes[0].set_title("Test Accuracy: Dùng vs Không dùng (càng cao càng tốt)")
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=20, ha="right")
axes[0].set_ylabel("Test Accuracy"); axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(x - w / 2, off_gap, w, label="Không dùng", color="#d95f5f")
axes[1].bar(x + w / 2, on_gap, w, label="Dùng", color="#5f9ed9")
axes[1].set_title("Overfitting Gap: Dùng vs Không dùng (càng thấp càng tốt)")
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20, ha="right")
axes[1].set_ylabel("Gap (Train - Val)"); axes[1].legend(); axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()